# 02 · OOP, typing and Pydantic

**Estimated study time: 12 minutes.**

You already know programming from Java. Here we focus on what feels different in Python. Run the cells in order, predict the results, and change one value before moving on.

## Dataclasses vs validated DTOs

Think of `@dataclass` as concise data-carrier boilerplate, somewhat like a Java record. Pydantic `BaseModel` additionally performs runtime validation when creating instances. Type hints alone do **not** validate input.

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel, Field, ValidationError

@dataclass
class AuditEntry:
    ticket_id: int
    action: str

class TicketInput(BaseModel):
    ticket_id: int = Field(gt=0)
    text: str = Field(min_length=3)
    priority: str = "normal"

item = TicketInput(ticket_id=12, text="Cannot access account")
print(item.model_dump())
try:
    TicketInput(ticket_id=-3, text="no")
except ValidationError as exc:
    print("Validation rejected invalid input:", len(exc.errors()), "errors")

## Protocols: interface-like contracts

A `Protocol` describes required methods structurally. Unlike Java interfaces, a class need not explicitly declare implementation.

In [ ]:
from typing import Protocol

class Formatter(Protocol):
    def format(self, ticket_id: int) -> str: ...

class TicketFormatter:
    def format(self, ticket_id: int) -> str:
        return f"T-{ticket_id}"

formatter: Formatter = TicketFormatter()
print(formatter.format(42))

## Decorators: functions that wrap or register other functions

Java annotations and Python decorators can serve similar high-level use cases, but a Python decorator actually receives and may replace a function at definition time. Agent frameworks commonly use decorators to register tools.

In [ ]:
from functools import wraps

def log_call(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("Calling", fn.__name__)
        return fn(*args, **kwargs)
    return wrapper

@log_call
def lookup_ticket(ticket_id: int) -> str:
    return f"Ticket {ticket_id}"

assert lookup_ticket(3) == "Ticket 3"

### Checkpoint

Change `TicketInput` so `priority` must be one of `low`, `normal`, `high` using `Literal`. A solution is below.

In [ ]:
from typing import Literal
class StrictTicketInput(BaseModel):
    ticket_id: int = Field(gt=0)
    priority: Literal["low", "normal", "high"] = "normal"

assert StrictTicketInput(ticket_id=1, priority="high").priority == "high"